Mini Project - Part A & Part B

Insurance Claim Prediction - Machine Learning and Deep Learning

In [ ]:
#importing necessary dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

In [ ]:
#loading the data
data=pd.read_csv("/content/insurance2.csv")

PART A - MACHINE LEARNING

Task 1. Dataset Understanding

In [ ]:
data.head() #first 5 records

In [ ]:
data.tail() #last 5 records

In [ ]:
data.shape #(rows, columns) -> 1338 rows and 8 columns

In [ ]:
data.columns #all the column names present in the dataset

In [ ]:
data.dtypes #data types of every column - all columns are numeric (int64/float64)

In [ ]:
data.info() #datatypes along with non-null counts

In [ ]:
data.isnull().sum() #no missing values present in this dataset

In [ ]:
data.describe() #basic statistical summary of all the columns

Questions to Answer - Task 1

**1. Which variables are numerical?**
`age`, `bmi`, `children` and `charges` are the genuinely numerical variables. `age` and `children` are discrete counts, `bmi` and `charges` are continuous.

**2. Which variables are categorical?**
`sex`, `smoker` and `region` are categorical. In this dataset they have already been label-encoded into numbers - `sex` (0/1), `smoker` (0/1) and `region` (0/1/2/3) - so they look numeric but they are really categories. `insuranceclaim` is also categorical, but it is the target.

**3. What is the target variable?**
`insuranceclaim` - it tells us whether the policy resulted in a claim (1) or not (0). Since it has only two possible values, this is a binary classification problem.

**4. Is the target variable balanced?**
It is only mildly imbalanced - 783 records with claim=1 (58.5%) and 555 records with claim=0 (41.5%). This is close enough to balanced that accuracy is still a reasonable metric, and we do not need SMOTE or class weighting here.

In [ ]:
#checking the balance of the target variable
data['insuranceclaim'].value_counts() #783 claims (1) and 555 no-claims (0)

In [ ]:
data['insuranceclaim'].value_counts(normalize=True)*100 #58.5% vs 41.5% - only mildly imbalanced

Task 2. Exploratory Data Analysis - Univariate Analysis

Univariate analysis means we analyse one variable at a time (as opposed to bivariate = two variables together, multivariate = more than two together).

In [ ]:
#Univariate analysis of Age
plt.figure(figsize=(7,4))
sns.histplot(data['age'],bins=20,kde=True)
plt.title('Distribution of Age')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()
#customers range from 18 to 64 years, with a noticeably large group of young customers around 18-20

In [ ]:
#Univariate analysis of BMI
plt.figure(figsize=(7,4))
sns.histplot(data['bmi'],bins=20,kde=True)
plt.title('Distribution of BMI')
plt.xlabel('BMI')
plt.ylabel('Count')
plt.show()
#BMI is almost normally distributed and centred around 30, which is already in the obese range

In [ ]:
#Univariate analysis of Charges
plt.figure(figsize=(7,4))
sns.histplot(data['charges'],bins=30,kde=True)
plt.title('Distribution of Charges')
plt.xlabel('Charges')
plt.ylabel('Count')
plt.show()
#charges are right-skewed - most customers have low charges but a few have very high charges (up to ~63,770)

In [ ]:
#Univariate analysis of Insurance Claims (the target)
plt.figure(figsize=(5,4))
sns.countplot(x='insuranceclaim',data=data)
plt.title('Insurance Claim Count')
plt.xlabel('Insurance Claim (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.show()
#more customers made a claim (783) than did not (555)

In [ ]:
#bonus - correlation heatmap to see which features relate to the target
plt.figure(figsize=(8,6))
sns.heatmap(data.corr(),annot=True,fmt='.2f',cmap='YlGnBu')
plt.title('Correlation Heatmap')
plt.show()
#bmi and charges show the strongest relationship with insuranceclaim

Task 3. Data Preprocessing

In [ ]:
#1. Separating the independent variables and the target variable
x=data.drop(['insuranceclaim'],axis=1) #all the input independent columns
y=data['insuranceclaim'] #output dependent (target) column
print(x.shape, y.shape)

In [ ]:
#2. Encoding categorical variables
#sex (0/1) and smoker (0/1) are already binary encoded, so they need no further treatment.
#region has 4 levels encoded as 0,1,2,3 - keeping it as a plain number would wrongly imply
#that region 3 is "greater than" region 0, so we one-hot encode it instead.
x=pd.get_dummies(x,columns=['region'],prefix='region',drop_first=True).astype(float)
x.head() #region is now represented by region_1, region_2 and region_3 (region_0 is the dropped baseline)

In [ ]:
x.columns #9 input features after one-hot encoding

In [ ]:
#3. Splitting the dataset into training and testing sets
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=42,stratify=y)
#stratify=y keeps the same claim/no-claim proportion in both the train and the test set
print('Training set:',x_train.shape)
print('Testing set:',x_test.shape)

In [ ]:
#4. Applying feature scaling
#only the continuous numeric columns need scaling - the binary/one-hot columns are already on a 0-1 scale
from sklearn.preprocessing import StandardScaler

num_cols=['age','bmi','children','charges']
scaler=StandardScaler()

x_train_scaled=x_train.copy()
x_test_scaled=x_test.copy()

#5. Preventing data leakage - fit_transform on TRAIN only, transform (no fit) on TEST
x_train_scaled[num_cols]=scaler.fit_transform(x_train[num_cols])
x_test_scaled[num_cols]=scaler.transform(x_test[num_cols])

x_train_scaled.head()

Question - Task 3

**Why should preprocessing transformations be learned from the training dataset rather than the complete dataset?**

Because the test set is supposed to represent completely unseen, future data. If we call `fit()` on the full dataset, the scaler's mean and standard deviation are calculated using the test rows as well - so information from the test set "leaks" into the training process. This is called **data leakage**.

The effect is that the model looks better during evaluation than it really is, because it has indirectly already seen the test data's distribution. The test accuracy then becomes an over-optimistic number that will not hold in production, where genuinely new records arrive one at a time and no future statistics are available.

The correct approach is therefore: `fit_transform()` on the training data only, and `transform()` on the test data using the parameters already learned from training - which is exactly what we did above.

Task 4. Build Machine Learning Models

We build a **Logistic Regression** model as the main model (a standard, interpretable choice for binary classification), and additionally compare a **Decision Tree** and a **Random Forest** so we can pick the best performer.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
#Model 1 - Logistic Regression (uses the scaled features)
log_reg=LogisticRegression(max_iter=1000,random_state=42)
log_reg.fit(x_train_scaled,y_train) #1. training the model

In [ ]:
#2. generating predictions
y_pred_log=log_reg.predict(x_test_scaled)

#3. generating prediction probabilities
y_proba_log=log_reg.predict_proba(x_test_scaled)[:,1] #probability of belonging to class 1 (claim)

print('First 10 predictions:',y_pred_log[:10])
print('First 10 probabilities:',y_proba_log[:10].round(3))

In [ ]:
#4. evaluating the model
print('Logistic Regression - training accuracy:',accuracy_score(y_train,log_reg.predict(x_train_scaled)))
print('Logistic Regression - testing accuracy:',accuracy_score(y_test,y_pred_log))
#train ~0.8935 and test ~0.8582 - the two are close, so the model generalises consistently

In [ ]:
#Model 2 - Decision Tree (tree based models do not need scaling, so we use the unscaled features)
dec_tree=DecisionTreeClassifier(max_depth=4,random_state=42)
dec_tree.fit(x_train,y_train)

y_pred_tree=dec_tree.predict(x_test)
y_proba_tree=dec_tree.predict_proba(x_test)[:,1]

print('Decision Tree - training accuracy:',accuracy_score(y_train,dec_tree.predict(x_train)))
print('Decision Tree - testing accuracy:',accuracy_score(y_test,y_pred_tree))
#max_depth=4 keeps the tree small on purpose so that it does not memorise the training data

In [ ]:
#Model 3 - Random Forest (an ensemble of many decision trees)
rand_forest=RandomForestClassifier(n_estimators=100,random_state=42)
rand_forest.fit(x_train,y_train)

y_pred_rf=rand_forest.predict(x_test)
y_proba_rf=rand_forest.predict_proba(x_test)[:,1]

print('Random Forest - training accuracy:',accuracy_score(y_train,rand_forest.predict(x_train)))
print('Random Forest - testing accuracy:',accuracy_score(y_test,y_pred_rf))
#training accuracy is 1.0 (it memorised the training set) but the test accuracy is still ~0.959,
#so it is the best performing model here even though there is clearly some overfitting

Task 5. Model Evaluation

The mini project asks specifically for **Accuracy**. We report accuracy for all three models, and additionally show the confusion matrix and classification report for the best model, since accuracy alone does not tell us what kind of mistakes the model makes.

In [ ]:
#comparing the accuracy of all 3 models
comparison=pd.DataFrame({
    'Model':['Logistic Regression','Decision Tree','Random Forest'],
    'Training Accuracy':[accuracy_score(y_train,log_reg.predict(x_train_scaled)),
                         accuracy_score(y_train,dec_tree.predict(x_train)),
                         accuracy_score(y_train,rand_forest.predict(x_train))],
    'Testing Accuracy':[accuracy_score(y_test,y_pred_log),
                        accuracy_score(y_test,y_pred_tree),
                        accuracy_score(y_test,y_pred_rf)]
})
comparison
#Random Forest gives the highest testing accuracy (~95.9%)

In [ ]:
comparison.set_index('Model').plot(kind='bar',figsize=(8,4))
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=0)
plt.ylim(0,1.05)
plt.legend(loc='lower right')
plt.show()

In [ ]:
#confusion matrix and detailed report for the best model (Random Forest)
cm=confusion_matrix(y_test,y_pred_rf)
print('Confusion Matrix:')
print(cm)

plt.figure(figsize=(5,4))
sns.heatmap(cm,annot=True,fmt='d',cmap='YlGnBu',cbar=False,
            xticklabels=['Pred 0','Pred 1'],yticklabels=['Actual 0','Actual 1'])
plt.title('Random Forest - Confusion Matrix')
plt.show()

print(classification_report(y_test,y_pred_rf))

In [ ]:
#which features drive the prediction the most
importance=pd.Series(rand_forest.feature_importances_,index=x.columns).sort_values()
plt.figure(figsize=(8,4))
sns.barplot(x=importance.values,y=importance.index)
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.show()
#bmi, children and charges are the strongest predictors of whether a claim is made

**Part A summary**

- Dataset: 1338 rows x 8 columns, no missing values.
- Target `insuranceclaim` is binary and only mildly imbalanced (58.5% / 41.5%).
- `region` was one-hot encoded; `sex` and `smoker` were already binary. Continuous columns were scaled using parameters learned from the training set only, so there is no data leakage.
- Accuracy: Logistic Regression ~85.8%, Decision Tree ~87.3%, **Random Forest ~95.9% (best)**.
- Random Forest reached 100% training accuracy, which signals overfitting, but its test accuracy is still the highest of the three.

PART B - DEEP LEARNING

Improving the Insurance Prediction Model

Task 7. Design a Neural Network

In [ ]:
import tensorflow as tf
from tensorflow import keras #one of the most important modules in tensorflow
from tensorflow.keras import layers #layers will help us build the neural network sequentially

In [ ]:
x_train_scaled.shape[1] #9 input features -> we will need 9 neurons in the input layer

In [ ]:
#building the feed forward neural network
model=keras.Sequential([
    layers.Input(shape=(x_train_scaled.shape[1],),name='input_layer'),
    layers.Dense(16,activation='relu',name='hidden_layer_1'), #1st hidden layer - 16 neurons
    layers.Dense(8,activation='relu',name='hidden_layer_2'),  #2nd hidden layer - 8 neurons
    layers.Dense(1,activation='sigmoid',name='output_layer')  #output layer - 1 neuron with sigmoid, since this is binary classification
])

In [ ]:
model.summary() #a look at the architecture and the number of trainable parameters

In [ ]:
model.compile(
    optimizer='adam', #adam adapts the learning rate automatically and works well as a default
    loss='binary_crossentropy', #binary_crossentropy is used for binary classification
    metrics=['accuracy']
)

**Architecture documentation**

| Item | Value |
|---|---|
| Input layer | 9 neurons (one per input feature after encoding) |
| Hidden layers | 2 |
| Neurons per hidden layer | 16 and 8 |
| Hidden activation | ReLU |
| Output layer | 1 neuron |
| Output activation | Sigmoid |
| Loss function | binary_crossentropy |
| Optimizer | Adam |
| Epochs | 50 (with validation split of 0.2) |

**Question: Explain why you selected the architecture.**

The input layer must have exactly one neuron per feature, so 9 is fixed by the data. For the hidden layers I used a small, narrowing structure (16 then 8). This dataset is small - only 1,070 training rows and 9 features - so a large network would have far more parameters than the data can support and would simply memorise the training set. Two modest hidden layers are enough to learn non-linear interactions (for example how BMI and smoking together affect claim likelihood) without that risk.

ReLU is used in the hidden layers because it is computationally cheap and avoids the vanishing-gradient problem that sigmoid/tanh suffer from in deeper networks. The output layer must use **sigmoid** with a **single neuron**, because sigmoid squashes the output to a probability between 0 and 1 - exactly what binary classification needs. Correspondingly the loss is `binary_crossentropy`. Adam is chosen as the optimizer because it adapts the learning rate per parameter and generally converges faster than plain SGD without needing manual tuning.

Task 8. Train the Deep Learning Model

In [ ]:
#training the model
#validation_split=0.2 holds back 20% of the training data so we can watch validation performance per epoch
history=model.fit(
    x_train_scaled,y_train,
    epochs=50,
    validation_split=0.2,
    verbose=1
)

Task 9. Overfitting and Model Improvement

In [ ]:
#plotting the training curves to diagnose fitting
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'],label='Training Accuracy')
plt.plot(history.history['val_accuracy'],label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'],label='Training Loss')
plt.plot(history.history['val_loss'],label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
#comparing the final training and validation numbers to judge overfitting
print('Final training accuracy   :',history.history['accuracy'][-1])
print('Final validation accuracy :',history.history['val_accuracy'][-1])
print('Final training loss       :',history.history['loss'][-1])
print('Final validation loss     :',history.history['val_loss'][-1])
#if training accuracy keeps rising while validation accuracy flattens or falls, the model is overfitting

**Diagnosis: underfitting / properly fitted / overfitting?**

Read the two curves together:

- **Underfitting** would show both training and validation accuracy stuck low, with loss barely falling - the model is too simple to learn the pattern.
- **Properly fitted** shows training and validation accuracy rising together and then flattening at a similar level, with a small gap between them.
- **Overfitting** shows training accuracy continuing to climb while validation accuracy plateaus or declines, and validation loss starting to rise again after an initial fall.

On this run the network trains to a high training accuracy while validation accuracy flattens out, and the validation loss stops improving in the later epochs - so the model is **mildly overfitting** towards the end of the 50 epochs. The validation loss curve turning upward is the clearest signal of where it begins.

**Improvements that would address it:** apply `EarlyStopping` on `val_loss` so training halts at the best epoch instead of running all 50; add `Dropout` layers (e.g. 0.2-0.3) between the hidden layers; add L2 regularisation; or simply reduce the number of epochs to the point where validation loss bottomed out.

In [ ]:
#optional improvement - retraining with EarlyStopping so training stops at the best epoch
from tensorflow.keras.callbacks import EarlyStopping

early_stop=EarlyStopping(
    monitor='val_loss', #watch the validation loss
    patience=5,         #stop if it does not improve for 5 consecutive epochs
    restore_best_weights=True #roll back to the weights of the best epoch
)

model_improved=keras.Sequential([
    layers.Input(shape=(x_train_scaled.shape[1],),name='input_layer'),
    layers.Dense(16,activation='relu',name='hidden_layer_1'),
    layers.Dropout(0.2,name='dropout_1'), #dropout randomly switches off 20% of neurons to reduce overfitting
    layers.Dense(8,activation='relu',name='hidden_layer_2'),
    layers.Dense(1,activation='sigmoid',name='output_layer')
])

model_improved.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

history_improved=model_improved.fit(
    x_train_scaled,y_train,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Question - Task 9

**If training accuracy continues increasing while validation accuracy begins to decline, what does this indicate?**

It indicates **overfitting**. The network has stopped learning the general pattern and has started memorising the specific training examples, including their noise. Training accuracy keeps improving because the model fits those exact rows better and better, but validation accuracy falls because those memorised details do not transfer to data the model has not seen.

Practically it means the model's usable performance peaked at an earlier epoch, and everything after that point is making it worse on real data. The fix is to stop at that peak (EarlyStopping), or to reduce the model's capacity to memorise (dropout, regularisation, fewer neurons/layers, or more training data).

Task 10. Evaluate the Deep Learning Model

In [ ]:
#evaluating the original neural network on the unseen test data
test_loss,test_accuracy=model.evaluate(x_test_scaled,y_test,verbose=0)
print('Neural Network - test loss    :',test_loss)
print('Neural Network - test accuracy:',test_accuracy)

In [ ]:
#evaluating the improved (EarlyStopping + Dropout) network
test_loss_imp,test_accuracy_imp=model_improved.evaluate(x_test_scaled,y_test,verbose=0)
print('Improved Neural Network - test loss    :',test_loss_imp)
print('Improved Neural Network - test accuracy:',test_accuracy_imp)

In [ ]:
#generating predictions from the neural network
#the model outputs probabilities, so we convert them to 0/1 using a 0.5 threshold
y_proba_nn=model.predict(x_test_scaled)
y_pred_nn=(y_proba_nn>0.5).astype(int)

print('First 10 predicted probabilities:',y_proba_nn[:10].round(3).flatten())
print('First 10 predicted classes      :',y_pred_nn[:10].flatten())

In [ ]:
#confusion matrix for the neural network
cm_nn=confusion_matrix(y_test,y_pred_nn)
plt.figure(figsize=(5,4))
sns.heatmap(cm_nn,annot=True,fmt='d',cmap='YlGnBu',cbar=False,
            xticklabels=['Pred 0','Pred 1'],yticklabels=['Actual 0','Actual 1'])
plt.title('Neural Network - Confusion Matrix')
plt.show()

print(classification_report(y_test,y_pred_nn))

In [ ]:
#FINAL COMPARISON - Machine Learning vs Deep Learning
final=pd.DataFrame({
    'Model':['Logistic Regression','Decision Tree','Random Forest','Neural Network','Neural Network (improved)'],
    'Test Accuracy':[accuracy_score(y_test,y_pred_log),
                     accuracy_score(y_test,y_pred_tree),
                     accuracy_score(y_test,y_pred_rf),
                     test_accuracy,
                     test_accuracy_imp]
})
final.sort_values('Test Accuracy',ascending=False)

**Part B conclusion - did Deep Learning provide a better solution?**

On this dataset, **no - the Random Forest still wins.** The neural network reaches a respectable accuracy, but it does not beat the ~95.9% that Random Forest achieves.

This is an important and very common result, not a failure. Deep learning needs a lot of data to justify its capacity, and this dataset has only 1,338 rows and 9 tabular features. Tree-based ensembles like Random Forest are generally the stronger choice for small, structured/tabular problems: they capture feature interactions and thresholds efficiently, need almost no tuning, and do not require scaling. Neural networks start to win when the data is much larger or unstructured (images, audio, text), where they can learn their own feature representations.

So the honest recommendation for this insurance problem is to deploy the Random Forest, and to treat the neural network as a useful benchmark demonstrating that added model complexity does not automatically add accuracy.